# Lesson 1.6 — Model Selection, Batch vs Realtime, and Token Cost

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Choose among Opus, Sonnet, and Haiku by quality/latency/cost
- Decide between realtime Messages API and the Message Batches API
- Build a token cost calculator from usage data
- Account for breaking behavior changes across model releases

**You will produce:** A model-selection helper, a batch-vs-realtime decision function, and a token cost calculator.

## Concepts

**Model tiers** trade capability for speed and price:

- **Opus**: highest capability for the hardest reasoning/agentic tasks; most expensive.
- **Sonnet**: balanced default for most production workloads.
- **Haiku**: fastest and cheapest for high-volume, simpler tasks.

Pick the smallest tier that meets quality needs, and **pin model versions** so a new release does not silently change behavior (a real exam concern: breaking behavior changes across releases).

**Batch vs realtime**: the **Message Batches API** processes large, latency-tolerant workloads asynchronously within a ~24-hour window at reduced cost - ideal for overnight bulk jobs (this is exactly Sample Question 1). Use the realtime Messages API when a user is waiting.

**Cost modeling**: cost = input_tokens x input_rate + output_tokens x output_rate, adjusted for cache writes/reads and batch discounts. Track `usage` on every response.

> **Reference:** [Anthropic - Models overview & pricing](https://docs.anthropic.com/en/docs/about-claude/models)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A model-selection helper

Encode the quality/latency/cost tradeoff as a simple policy.

In [ ]:
def select_model(task):
    if task.get("hard_reasoning"):
        return MODEL_OPUS
    if task.get("high_volume") and not task.get("needs_nuance"):
        return MODEL_HAIKU
    return MODEL_SONNET

print(select_model({"hard_reasoning": True}))
print(select_model({"high_volume": True}))
print(select_model({}))

## Code-Along: Step 2 — Batch vs realtime decision

Map requirements to the right API (mirrors Sample Question 1).

In [ ]:
def choose_api(volume, deadline_hours, user_waiting):
    if user_waiting:
        return "realtime Messages API"
    if volume >= 1000 and deadline_hours >= 1:
        return "Message Batches API (cheaper, async, ~24h window)"
    return "realtime Messages API"

print(choose_api(volume=10000, deadline_hours=12, user_waiting=False))

## Code-Along: Step 3 — Token cost calculator

Compute cost from usage. Rates are illustrative - confirm current pricing in the docs.

In [ ]:
# Illustrative $ per 1M tokens (CONFIRM at https://docs.anthropic.com/en/docs/about-claude/pricing)
RATES = {
    MODEL_HAIKU: {"in": 1.00, "out": 5.00},
    MODEL_SONNET: {"in": 3.00, "out": 15.00},
    MODEL_OPUS: {"in": 15.00, "out": 75.00},
}

def cost(model, input_tokens, output_tokens, batch=False):
    r = RATES[model]
    c = (input_tokens / 1e6) * r["in"] + (output_tokens / 1e6) * r["out"]
    return c * 0.5 if batch else c  # batch often ~50% off; confirm in docs

print("Realtime:", cost(MODEL_SONNET, 500_000, 100_000))
print("Batch:   ", cost(MODEL_SONNET, 500_000, 100_000, batch=True))

## Exercise

Extend the cost calculator into `monthly_estimate(model, reqs_per_day, in_tok, out_tok, batch=False)` that returns projected 30-day spend. Then compare Haiku vs Sonnet for 50k requests/day.

In [ ]:
def monthly_estimate(model, reqs_per_day, in_tok, out_tok, batch=False):
    # TODO: per-request cost via cost(...)
    # TODO: multiply by reqs_per_day * 30
    pass

# TODO: print Haiku vs Sonnet monthly estimates for a high-volume job

## Key Takeaways

- Pick the smallest model tier that meets quality; pin versions to avoid breaking changes.
- Use the Message Batches API for large, non-urgent jobs to cut cost (Sample Q1 answer: B).
- Use realtime Messages API whenever a user is waiting on the result.
- Model cost = input/output tokens x rates, adjusted for caching and batch discounts.
- Track usage on every response to keep cost models honest.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.